In [7]:
import os
import glob
import joblib
import itertools
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression

In [8]:
class ImprovedLayeredClassifier(BaseEstimator, ClassifierMixin):
    """
    Enhanced two-layer classifier with multiple improvements:
    - Class-weighted logistic regression
    - Optimizable soft routing weights
    - Better feature representation
    - Threshold calibration
    """
    
    def __init__(self, max_features=15000, ngram_range=(1, 3), 
                 routing_strategy='soft', class_weight='balanced', random_state=42):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.random_state = random_state
        self.routing_strategy = routing_strategy  # 'soft', 'hard', 'weighted', 'ensemble'
        self.class_weight = class_weight
        
        # Layer 1: Group classifier (with class weights)
        self.layer1_classifier = LogisticRegression(
            max_iter=1000, 
            random_state=random_state,
            class_weight=class_weight,
            solver='lbfgs'
        )
        
        # Layer 2: Fine-grained classifiers (with class weights)
        self.layer2a_classifier = LogisticRegression(
            max_iter=1000, 
            random_state=random_state,
            class_weight=class_weight,
            solver='lbfgs'
        )  # For 1/2/3
        self.layer2b_classifier = LogisticRegression(
            max_iter=1000, 
            random_state=random_state,
            class_weight=class_weight,
            solver='lbfgs'
        )  # For 3/4/5
        
        # Vectorizers with enhanced n-grams
        self.layer1_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,  # Ignore very rare terms
            max_df=0.95  # Ignore very common terms
        )
        self.layer2a_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,
            max_df=0.95
        )
        self.layer2b_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,
            max_df=0.95
        )
        
        # Routing weights (can be optimized)
        self.routing_weights = None
    
    def fit(self, X, y, X_val=None, y_val=None):
        """
        Train the improved layered classifier.
        
        Args:
            X: Series of text reviews
            y: Series of star ratings (1-5)
            X_val: Optional validation set for routing optimization
            y_val: Optional validation labels
        """
        if not isinstance(X, pd.Series):
            X = pd.Series(X)
        if not isinstance(y, pd.Series):
            y = pd.Series(y)
        
        print("\n" + "="*80)
        print("TRAINING IMPROVED LAYERED CLASSIFIER")
        print(f"Strategy: {self.routing_strategy.upper()} | Class Weight: {self.class_weight}")
        print("="*80)
        
        # --- LAYER 1: BINARY GROUP CLASSIFIER ---
        # print("\n[LAYER 1] Training binary group classifier (1/2/3 vs 3/4/5)")
        # print("-" * 80)
        
        group_labels_list = []
        X_list = []
        
        mask_123 = y <= 3
        X_123 = X[mask_123]
        X_list.append(X_123)
        group_labels_list.append(np.zeros(len(X_123), dtype=int))
        
        mask_345 = y >= 3
        X_345 = X[mask_345]
        X_list.append(X_345)
        group_labels_list.append(np.ones(len(X_345), dtype=int))
        
        X_layer1 = pd.concat(X_list, ignore_index=True)
        group_labels = np.concatenate(group_labels_list)
        
        X_layer1_tfidf = self.layer1_vectorizer.fit_transform(X_layer1)
        self.layer1_classifier.fit(X_layer1_tfidf, group_labels)
        
        layer1_acc = accuracy_score(
            group_labels, self.layer1_classifier.predict(X_layer1_tfidf)
        )
        print(f"  Training Accuracy: {layer1_acc:.4f}")
        print(f"  Samples: {len(X_layer1)} (Group 0: {(group_labels==0).sum()}, "
              f"Group 1: {(group_labels==1).sum()})")
        
        # --- LAYER 2A: FINE CLASSIFIER FOR 1/2/3 ---
        # print("\n[LAYER 2A] Training fine classifier for 1/2/3 stars")
        # print("-" * 80)
        
        mask_123_fine = y <= 3
        X_123_fine = X[mask_123_fine]
        y_123_fine = y[mask_123_fine]
        
        X_123_tfidf = self.layer2a_vectorizer.fit_transform(X_123_fine)
        self.layer2a_classifier.fit(X_123_tfidf, y_123_fine)
        
        layer2a_acc = accuracy_score(
            y_123_fine, self.layer2a_classifier.predict(X_123_tfidf)
        )
        print(f"  Training Accuracy: {layer2a_acc:.4f}")
        print(f"  Samples: {len(X_123_fine)} (1: {(y_123_fine==1).sum()}, "
              f"2: {(y_123_fine==2).sum()}, 3: {(y_123_fine==3).sum()})")
        
        # --- LAYER 2B: FINE CLASSIFIER FOR 3/4/5 ---
        # print("\n[LAYER 2B] Training fine classifier for 3/4/5 stars")
        # print("-" * 80)
        
        mask_345_fine = y >= 3
        X_345_fine = X[mask_345_fine]
        y_345_fine = y[mask_345_fine]
        
        X_345_tfidf = self.layer2b_vectorizer.fit_transform(X_345_fine)
        self.layer2b_classifier.fit(X_345_tfidf, y_345_fine)
        
        layer2b_acc = accuracy_score(
            y_345_fine, self.layer2b_classifier.predict(X_345_tfidf)
        )
        print(f"  Training Accuracy: {layer2b_acc:.4f}")
        print(f"  Samples: {len(X_345_fine)} (3: {(y_345_fine==3).sum()}, "
              f"4: {(y_345_fine==4).sum()}, 5: {(y_345_fine==5).sum()})")
        
        # Optional: Optimize routing weights on validation set
        if X_val is not None and y_val is not None:
            print("\n[OPTIMIZATION] Optimizing routing weights on validation set...")
            self._optimize_routing_weights(X_val, y_val)
        else:
            self.routing_weights = np.array([0.5, 0.5])  # Default equal weights
        
        print("\n" + "="*80)
        print("Training Complete")
        print("="*80 + "\n")
        
        return self
    
    def _optimize_routing_weights(self, X_val, y_val):
        """
        Optimize the soft routing weights using validation set.
        """
        def objective(w1):
            w = np.array([w1, 1 - w1])
            self.routing_weights = w
            preds = self.predict(X_val)
            return -f1_score(y_val, preds, average='weighted')
        
        result = minimize_scalar(objective, bounds=(0.2, 0.8), method='bounded')
        self.routing_weights = np.array([result.x, 1 - result.x])
        print(f"  Optimized weights: Group0={self.routing_weights[0]:.4f}, "
              f"Group1={self.routing_weights[1]:.4f}")
    
    def predict_proba(self, X):
        """
        Predict probabilities using the specified routing strategy.
        """
        if not isinstance(X, pd.Series):
            X = pd.Series(X)
        
        # Layer 1 predictions
        X_layer1_tfidf = self.layer1_vectorizer.transform(X)
        group_proba = self.layer1_classifier.predict_proba(X_layer1_tfidf)
        
        # Layer 2 predictions
        X_layer2a_tfidf = self.layer2a_vectorizer.transform(X)
        X_layer2b_tfidf = self.layer2b_vectorizer.transform(X)
        
        proba_123 = self.layer2a_classifier.predict_proba(X_layer2a_tfidf)
        proba_345 = self.layer2b_classifier.predict_proba(X_layer2b_tfidf)
        
        classes_123 = self.layer2a_classifier.classes_
        classes_345 = self.layer2b_classifier.classes_
        
        # Initialize combined probabilities
        combined_proba = np.zeros((len(X), 5))
        
        if self.routing_strategy == 'soft':
            # Soft routing: weighted combination using group probabilities
            weights = self.routing_weights if self.routing_weights is not None else np.array([0.5, 0.5])
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += weights[0] * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += weights[1] * proba_345[:, i]
        
        elif self.routing_strategy == 'dynamic':
            # Dynamic routing: use actual group probabilities as weights
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += group_proba[:, 0] * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += group_proba[:, 1] * proba_345[:, i]
        
        elif self.routing_strategy == 'hard':
            # Hard routing: pick one path based on group prediction
            group_preds = self.layer1_classifier.predict(X_layer1_tfidf)
            for i in range(len(X)):
                if group_preds[i] == 0:
                    for j, class_label in enumerate(classes_123):
                        combined_proba[i, class_label - 1] = proba_123[i, j]
                else:
                    for j, class_label in enumerate(classes_345):
                        combined_proba[i, class_label - 1] = proba_345[i, j]
        
        elif self.routing_strategy == 'ensemble':
            # Ensemble: average both paths equally
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += 0.5 * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += 0.5 * proba_345[:, i]
        
        # Normalize
        combined_proba = combined_proba / combined_proba.sum(axis=1, keepdims=True)
        
        return combined_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1) + 1

In [9]:
def load_data(folder_path='../Preprocessing-FeatureExtraction/cleaned-data'):
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder_path}")
    
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    data = pd.concat(dfs, ignore_index=True)
    data = data.dropna(subset=['stars', 'clean_text'])
    data['stars'] = data['stars'].astype(int)
    return data

In [10]:
def stratified_manual_split(X, y, train_size=200000, val_size=25000, test_size=25000, random_state=42):
    total_size = min(train_size + val_size + test_size, len(X))
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    X_train_parts, X_val_parts, X_test_parts = [], [], []
    y_train_parts, y_val_parts, y_test_parts = [], [], []

    rng = np.random.RandomState(random_state)

    for star_class in sorted(y.unique()):
        mask = (y == star_class)
        X_class = X[mask].reset_index(drop=True)
        y_class = y[mask].reset_index(drop=True)

        n_class = len(y_class)
        if n_class == 0:
            continue

        n_train_c = max(15, int(n_class * train_size / total_size))
        n_val_c = max(5, int(n_class * val_size / total_size))
        n_test_c = max(5, n_class - n_train_c - n_val_c)

        indices = rng.permutation(n_class)

        X_train_parts.append(X_class.iloc[indices[:n_train_c]])
        y_train_parts.append(y_class.iloc[indices[:n_train_c]])

        val_end = n_train_c + n_val_c
        X_val_parts.append(X_class.iloc[indices[n_train_c:val_end]])
        y_val_parts.append(y_class.iloc[indices[n_train_c:val_end]])

        X_test_parts.append(X_class.iloc[val_end:val_end + n_test_c])
        y_test_parts.append(y_class.iloc[val_end:val_end + n_test_c])

    X_train = pd.concat(X_train_parts, ignore_index=True)
    y_train = pd.concat(y_train_parts, ignore_index=True)
    X_val = pd.concat(X_val_parts, ignore_index=True)
    y_val = pd.concat(y_val_parts, ignore_index=True)
    X_test = pd.concat(X_test_parts, ignore_index=True)
    y_test = pd.concat(y_test_parts, ignore_index=True)

    return X_train, y_train, X_val, y_val, X_test, y_test

In [11]:
def tune_hyperparameters(
    data,
    param_grid,
    train_size=200000,
    val_size=25000,
    test_size=25000,
    scoring='weighted_f1',   # 'weighted_f1' or 'accuracy' or 'macro_f1'
    results_csv='tuning_results_hard_routing.csv',
    best_model_path='best_improved_layered_hard.joblib',
    random_state=42
):
    # Prepare dataset
    total_size = min(train_size + val_size + test_size, len(data))
    print(f"Sampling {total_size} reviews for tuning...")
    data_sample = data.sample(n=total_size, random_state=random_state)
    X = data_sample['clean_text'].reset_index(drop=True)
    y = data_sample['stars'].astype(int).reset_index(drop=True)

    X_train, y_train, X_val, y_val, X_test, y_test = stratified_manual_split(
        X, y, train_size=train_size, val_size=val_size, test_size=test_size, random_state=random_state
    )

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    # ParameterGrid iteration
    grid = list(ParameterGrid(param_grid))
    print(f"Total combinations: {len(grid)}")

    rows = []
    best_score = -np.inf
    best_entry = None

    for params in tqdm(grid):
        # Build classifier with hard routing
        clf = ImprovedLayeredClassifier(
            max_features=params.get('max_features', 15000),
            ngram_range=params.get('ngram_range', (1,2)),
            routing_strategy='hard',
            class_weight=params.get('balanced'),
            random_state=random_state
        )

        min_df = params.get('min_df', 2)
        max_df = params.get('max_df', 0.95)
        max_features = params.get('max_features', 15000)
        ngram_range = params.get('ngram_range', (1,3))

        clf.layer1_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                                min_df=min_df, max_df=max_df)
        clf.layer2a_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                                 min_df=min_df, max_df=max_df)
        clf.layer2b_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                                 min_df=min_df, max_df=max_df)

        # Set logistic regression Cs if given
        c1 = params.get('C_layer1', 1.0)
        c2a = params.get('C_layer2a', 1.0)
        c2b = params.get('C_layer2b', 1.0)
        # Update classifier objects (before fit)
        clf.layer1_classifier.C = c1
        clf.layer2a_classifier.C = c2a
        clf.layer2b_classifier.C = c2b

        # Fit
        try:
            clf.fit(X_train, y_train)
        except Exception as e:
            print(f"Failed to fit for params {params}: {e}")
            continue

        # Validate
        y_val_pred = clf.predict(X_val)

        acc = accuracy_score(y_val, y_val_pred)
        weighted_f1 = f1_score(y_val, y_val_pred, average='weighted')
        macro_f1 = f1_score(y_val, y_val_pred, average='macro')

        report = classification_report(y_val, y_val_pred, labels=[1,2,3,4,5], output_dict=True, zero_division=0)

        if scoring == 'weighted_f1':
            score = weighted_f1
        elif scoring == 'macro_f1':
            score = macro_f1
        else:
            score = acc

        row = {
            **params,
            'val_accuracy': acc,
            'val_macro_f1': macro_f1,
            'val_weighted_f1': weighted_f1,
            'score': score,
            'report': report
        }
        rows.append(row)

        # Keep best
        if score > best_score:
            best_score = score
            best_entry = {
                'params': params,
                'clf': clf,
                'val_accuracy': acc,
                'val_macro_f1': macro_f1,
                'val_weighted_f1': weighted_f1
            }

    # Save results
    df_rows = []
    for r in rows:
        flat = {k: v for k, v in r.items() if k != 'report'}
        # Add per-class F1 if available
        rep = r.get('report', {})
        for star in [1,2,3,4,5]:
            key = f'f1_{star}'
            flat[key] = rep.get(str(star), {}).get('f1-score', np.nan)
        df_rows.append(flat)

    results_df = pd.DataFrame(df_rows)
    results_df.to_csv(results_csv, index=False)
    print(f"Saved tuning results to {results_csv}")

    # Save best model
    if best_entry is not None:
        joblib.dump(best_entry['clf'], best_model_path)
        print(f"Best params: {best_entry['params']}")
        print(f"Best validation score ({scoring}): {best_score:.4f}")
        print(f"Saved best model to {best_model_path}")

        # Evaluate best on test set and print summary
        best_clf = best_entry['clf']
        y_test_pred = best_clf.predict(X_test)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_weighted_f1 = f1_score(y_test, y_test_pred, average='weighted')
        test_macro_f1 = f1_score(y_test, y_test_pred, average='macro')

        print("\n=== BEST MODEL TEST EVAL ===")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Test Weighted-F1: {test_weighted_f1:.4f}")
        print(f"Test Macro-F1: {test_macro_f1:.4f}")

        test_report = classification_report(y_test, y_test_pred, labels=[1,2,3,4,5])
        print(test_report)

    else:
        print("No successful runs; no best model saved.")

    return results_df, best_entry

In [12]:
if __name__ == '__main__':
    # Load data
    data = load_data()

    # Define parameter grid to search over
    param_grid = {
        'max_features': [1000, 5000, 10000],
        'ngram_range': [(1,2)],
        'min_df': [2],
        'max_df': [0.95],
        'class_weight': ['balanced'],
        'C_layer1': [0.1, 1.0, 10.0],
        'C_layer2a': [0.1, 1.0, 10.0],
        'C_layer2b': [0.1, 1.0, 10.0]
    }

    # Run tuning
    results_df, best = tune_hyperparameters(
        data,
        param_grid=param_grid,
        train_size=200000,  
        val_size=25000,
        test_size=25000,
        scoring='weighted_f1',
        results_csv='tuning_results_hard_routing.csv',
        best_model_path='best_improved_layered_hard.joblib',
        random_state=42
    )

    print("\nTuning complete. Top 5 configurations by score:")
    print(results_df.sort_values('score', ascending=False).head(5))

Sampling 250000 reviews for tuning...
Train: 199998 | Val: 24997 | Test: 25005
Total combinations: 81


  0%|          | 0/81 [00:00<?, ?it/s]


TRAINING IMPROVED LAYERED CLASSIFIER
Strategy: HARD | Class Weight: None
  Training Accuracy: 0.8520
  Samples: 222477 (Group 0: 63664, Group 1: 158813)
  Training Accuracy: 0.6773
  Samples: 63664 (1: 24325, 2: 16860, 3: 22479)
  Training Accuracy: 0.6952
  Samples: 158813 (3: 22479, 4: 47516, 5: 88818)

Training Complete



  1%|          | 1/81 [01:21<1:49:10, 81.88s/it]


TRAINING IMPROVED LAYERED CLASSIFIER
Strategy: HARD | Class Weight: None
  Training Accuracy: 0.8629
  Samples: 222477 (Group 0: 63664, Group 1: 158813)
  Training Accuracy: 0.6997
  Samples: 63664 (1: 24325, 2: 16860, 3: 22479)
  Training Accuracy: 0.7172
  Samples: 158813 (3: 22479, 4: 47516, 5: 88818)

Training Complete



...


TRAINING IMPROVED LAYERED CLASSIFIER
Strategy: HARD | Class Weight: None
  Training Accuracy: 0.8714
  Samples: 222477 (Group 0: 63664, Group 1: 158813)
  Training Accuracy: 0.7495
  Samples: 63664 (1: 24325, 2: 16860, 3: 22479)
  Training Accuracy: 0.7413
  Samples: 158813 (3: 22479, 4: 47516, 5: 88818)

Training Complete



 99%|█████████▉| 80/81 [2:32:49<02:13, 133.45s/it]


TRAINING IMPROVED LAYERED CLASSIFIER
Strategy: HARD | Class Weight: None
  Training Accuracy: 0.8759
  Samples: 222477 (Group 0: 63664, Group 1: 158813)
  Training Accuracy: 0.7992
  Samples: 63664 (1: 24325, 2: 16860, 3: 22479)
  Training Accuracy: 0.7648
  Samples: 158813 (3: 22479, 4: 47516, 5: 88818)

Training Complete



100%|██████████| 81/81 [2:36:08<00:00, 115.66s/it]


Saved tuning results to tuning_results_hard_routing.csv
Best params: {'C_layer1': 10.0, 'C_layer2a': 1.0, 'C_layer2b': 1.0, 'class_weight': 'balanced', 'max_df': 0.95, 'max_features': 10000, 'min_df': 2, 'ngram_range': (1, 2)}
Best validation score (weighted_f1): 0.6577
Saved best model to best_improved_layered_hard.joblib

=== BEST MODEL TEST EVAL ===
Test Accuracy: 0.7179
Test Weighted-F1: 0.7078
Test Macro-F1: 0.6551
              precision    recall  f1-score   support

           1       0.77      0.84      0.80      3042
           2       0.62      0.46      0.53      2109
           3       0.59      0.51      0.55      2811
           4       0.61      0.52      0.57      5940
           5       0.78      0.89      0.83     11103

    accuracy                           0.72     25005
   macro avg       0.67      0.65      0.66     25005
weighted avg       0.71      0.72      0.71     25005


Tuning complete. Top 5 configurations by score:
    C_layer1  C_layer2a  C_layer2b cla